### Prompts

In [1]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import (StrOutputParser, 
                        CommaSeparatedListOutputParser, JsonOutputParser, PydanticOutputParser)
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnablePassthrough, RunnableGenerator, RunnableAssign
import os
from dotenv import load_dotenv

load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")

In [2]:
from langchain_core.prompts import (ChatPromptTemplate,  PromptTemplate, 
                                    ChatMessagePromptTemplate, HumanMessagePromptTemplate, SystemMessagePromptTemplate)
from langchain_core.output_parsers import JsonOutputParser
import json
from langchain_core.messages import HumanMessage, SystemMessage

# Initialize Groq LLM
llm = ChatOpenAI(
    model="gpt-4o",
    temperature=0.7
)
print(llm)

profile={'name': 'GPT-4o', 'release_date': '2024-05-13', 'last_updated': '2024-08-06', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True} client=<openai.resources.chat.completions.completions.Completions object at 0x000001680A66C250> async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001680BB4D9D0> root_client=<openai.OpenAI object at 0x000001680A65FD50> root_async_client=<openai.AsyncOpenAI object at 0x000001680BB4D4D0> model_name='gpt-4o' temperature=0.7 model_kwargs={} openai_api_key=SecretStr('*******

In [3]:
model = llm

In [4]:
from langchain_core.messages import HumanMessage,SystemMessage

# A manual way of creating messages
messages=[
    SystemMessage(content="Translate the following from English to French"),
    HumanMessage(content="Hello How are you?")
]
print(messages)

result=model.invoke(messages)
print(result)

[SystemMessage(content='Translate the following from English to French', additional_kwargs={}, response_metadata={}), HumanMessage(content='Hello How are you?', additional_kwargs={}, response_metadata={})]
content='Bonjour, comment ça va ?' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 6, 'prompt_tokens': 23, 'total_tokens': 29, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_88aa8039ec', 'id': 'chatcmpl-DXSQoLmkiWr4Om5RAAG4tEnzGnYmh', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019db57e-de09-7443-a3ae-95de30299ae4-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 23, 'output_tokens': 6, 'total_tokens': 29, 'input_token_details': {'audio': 

In [5]:
messages = [
    SystemMessage(content="Translate the following from English to {language}"),
    HumanMessage(content="{text}")
]
## Cant work

In [6]:
from langchain_core.messages import SystemMessage, HumanMessage

language = "French"
text = "Hello"

messages = [
    SystemMessage(content=f"Translate the following from English to {language}"),
    HumanMessage(content=text)
]
print(messages)

[SystemMessage(content='Translate the following from English to French', additional_kwargs={}, response_metadata={}), HumanMessage(content='Hello', additional_kwargs={}, response_metadata={})]


In [7]:
# solution 2: To use HumanMessagePromptTemplate and its counterpart
human = HumanMessagePromptTemplate.from_template("{input}")
system = SystemMessagePromptTemplate.from_template(
    "Translate the word from English to {language}"
)

prompt = ChatPromptTemplate.from_messages([system, human])

result = prompt.invoke({
    "input": "Hello, are you doing today",
    "language": "French"
})

print(result)

messages=[SystemMessage(content='Translate the word from English to French', additional_kwargs={}, response_metadata={}), HumanMessage(content='Hello, are you doing today', additional_kwargs={}, response_metadata={})]


In [8]:
chain = prompt | model 
chain.invoke({"input": "Hello, how are you doing today", "language": "Arabic"})

AIMessage(content='مرحبًا، كيف حالك اليوم؟', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 25, 'total_tokens': 34, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_a37395bab7', 'id': 'chatcmpl-DXSQpSdUOw5cwP08Tfnx3fwA9OiQw', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019db57e-e796-75f3-963b-50da5f9f86ce-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 25, 'output_tokens': 9, 'total_tokens': 34, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [9]:
chain.invoke({"input": "Hello, how are you doing today", "language": "Arabic"}).content

'مرحبًا، كيف حالك اليوم؟'

In [10]:
template = ChatPromptTemplate(
            [
                ("system", "Translate the word from Eglish to {language}."),
                ("human", "{input}"),
            ]
        )
template.invoke({
    "language": "Arabic",
    "input": "Hello, how are you doing, and how is family and work?"
})

ChatPromptValue(messages=[SystemMessage(content='Translate the word from Eglish to Arabic.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Hello, how are you doing, and how is family and work?', additional_kwargs={}, response_metadata={})])

In [11]:
chain = template | model

In [12]:
chain.invoke(
    {
    "language": "Arabic",
    "input": "Hello, how are you doing, and how is family and work?"
}
)

AIMessage(content='مرحبًا، كيف حالك، وكيف حال العائلة والعمل؟', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 35, 'total_tokens': 49, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_a37395bab7', 'id': 'chatcmpl-DXSQshkXZSsHhYIQsf1iCNe5mAjSJ', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019db57e-f10a-7192-9eca-ba5fd61ea094-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 35, 'output_tokens': 14, 'total_tokens': 49, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [13]:
chain.invoke(
    {
    "language": "Arabic",
    "input": "Hello, how are you doing, and how is family and work?"
}
).content

'مرحبًا، كيف حالك، وكيف حال العائلة والعمل؟'

In [14]:
chain.invoke(
    {
    "language": "Arabic",
    "input": "Hello, how are you doing, and how is family and work? It's great seeing you, I hope you are also"
}
).content

'مرحبا، كيف حالك وكيف حال العائلة والعمل؟ من الرائع رؤيتك، آمل أن تكون كذلك.'

In [15]:
template = ChatPromptTemplate(
            [
                ("system", "Write me very wonderful poem in {language}."),
                ("human", "{input}"),
            ]
        )
template.invoke({
    "language": "Arabic",
    "input": "Life"
})

ChatPromptValue(messages=[SystemMessage(content='Write me very wonderful poem in Arabic.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Life', additional_kwargs={}, response_metadata={})])

In [16]:
chain = template | model

In [17]:
print(chain.invoke({
    "language": "Arabic",
    "input": "Life"
}).content)

في دروب الحياة نسيرُ  
نحملُ الأحلامَ في قلوبٍ تُنيرُ  
تتشابكُ الأقدارُ مثلَ النهرِ  
تجري بنا نحوَ المجهولِ المستديرِ  

في الصباحِ نُغنّي للضوءِ  
وفي الليلِ نطلبُ من القمرِ الحضورَ  
نحيا بالحبِّ والأملِ  
ونزرعُ في الأرضِ بذورَ السرورِ  

بين الدموعِ والابتساماتِ  
تروي الحياةُ حكاياتُ البشرِ  
بعضنا يجدُ السعادةَ في البساطةِ  
وآخرون يبحثونَ عنها بينَ القصورِ  

لكنَّ الحياةَ ليست سوى لحظاتٍ  
تتأرجحُ بينَ الصبرِ والسرورِ  
فلنكنْ للعمرِ أصدقاءَ أوفياءَ  
ونحيا بحبٍّ لا يعرفُ الفتورَ  

في كلِّ غروبٍ وبزوغِ شمسٍ  
نجدُ في الحياةِ سحراً وجمالاً  
فهي رحلةٌ تتلوّنُ بالألوانِ  
وترسمُ في قلوبنا أعمقَ الآمالِ  

يا حياةُ، أنتِ الأملُ والمنالُ  
نحيا بكِ ونسعى لتحقيقِ المُحالِ  
فلنبقى على عهديكَ مخلصينَ  
نحملُ في قلوبنا حبَّ الزمانِ والوصالِ


In [18]:
template = ChatPromptTemplate(
            [
                ("system", "Write me very wonderful poem in {language}. After go immediately after completing the poem and then transalate into {language2}"),
                ("human", "{input}"),
            ]
        )
template.invoke({
    "language": "Arabic",
    "language2": "English",
    "input": "Life"
})

ChatPromptValue(messages=[SystemMessage(content='Write me very wonderful poem in Arabic. After go immediately after completing the poem and then transalate into English', additional_kwargs={}, response_metadata={}), HumanMessage(content='Life', additional_kwargs={}, response_metadata={})])

In [19]:
print(chain.invoke({
    "language": "Arabic",
    "language2": "English",
    "input": "Life"
}).content)

في دروب الحياة هناك أسرارُ  
تسري بنبض القلب والأفكارُ  

شموس تشرق في فضاء الأمل  
تضيء دروبنا، تزيل الأستارُ  

حين تبتسم الحياة كزهرة  
تفوح بالعطر، وتنساب الأقمارُ  

بين الفصول نعيش كالأشجار  
مرةً ننمو، ومرةً نحتارُ  

لكننا دوماً نعود لنزهر  
في وجه الرياح، نغني الأقدارُ  

الحياة لحنٌ يعزف الأماني  
ومعها نسير، نخطو كالأطيارُ  

فلنعيشها حباً وسلاماً  
ولنملأ الدروب بالحب والأنوارُ  


In [38]:
template = ChatPromptTemplate.from_messages([
    ("system", 
     "Write a beautiful poem in {language} about the given topic. "
     "Then provide a translation of the poem in {language2}. "
     "Format your response clearly with two sections:\n\n"
     "Poem ({language}):\n...\n\n"
     "Translation ({language2}):\n..."
    ),
    ("human", "{input}")
])

print(template.invoke({
    "language": "Arabic",
    "language2": "English",
    "input": "Life"
}).to_messages())

chain = template | model 

print(chain.invoke({
    "language": "Arabic",
    "language2": "English",
    "input": "Life"
}).content)

[SystemMessage(content='Write a beautiful poem in Arabic about the given topic. Then provide a translation of the poem in English. Format your response clearly with two sections:\n\nPoem (Arabic):\n...\n\nTranslation (English):\n...', additional_kwargs={}, response_metadata={}), HumanMessage(content='Life', additional_kwargs={}, response_metadata={})]
Poem (Arabic):
يا حياة، يا زهرة الأماني والآمال  
في كل صباح تشرقين كالآمال  
تحتضنين الأحلام في قلب الليالي  
وتنسجين الفرح بألوان الجمال  

يا حياة، يا نبع الحب والحنان  
تسري في عروقنا كنبض الأمان  
نغني لكِ في لحظات السعادة  
ونتوسل إليكِ في أوقات الحزن  

يا حياة، يا كتاب الأسرار العجيب  
نقرأ في صفحاتكِ كل يوم جديد  
تعلّميننا الصبر في وجه العواصف  
وتمنحيننا الأمل في عتمة الدروب  

يا حياة، يا لوحة الفن الرائعة  
تُرسمين بألوان الأمل والبهجة  
فيكِ نجد القوة لمواصلة الطريق  
وفيكِ نعيش الحب والأحلام  

Translation (English):
O life, O flower of wishes and hopes  
Every morning you rise like aspirations  
You embrace dreams in the h

In [21]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders.parsers import LLMImageBlobParser
from langchain_community.document_loaders.pdf import PyMuPDFParser, PyPDFLoader
from langchain_openai import ChatOpenAI

from langchain_community.document_loaders.pdf import PyPDFLoader

loader = PyPDFLoader(
    r"C:\Users\user\Downloads\Joins and set operations [Slides].pdf",
    mode="page"
)

docs = loader.load()

print(f"Total pages: {len(docs)}")
print(docs[2].page_content)

Total pages: 17
|
To better understand join and union operations, we use two datasets that are SDG 6 related: 
Countries table which has a Country_id, Country_name, and the countryʼs Population, and 
the Water_usage table which has data on Water_consumed_m3 listed per Country_id.
Joins and unions
3
The datasets
Country_id Country_name Population
1 Country A 1000000
2 Country B 1500000
3 Country C 5000000
4 Country D 2000000
Country_id Water_consumed_m3
1 500000
2 800000
3 3000000
5 1000000
Countries Water_usage


In [22]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
documents = text_splitter.split_documents(docs)
documents

[Document(metadata={'producer': 'PyPDF', 'creator': 'Google', 'creationdate': '', 'title': 'Joins and set operations [Slides]', 'source': 'C:\\Users\\user\\Downloads\\Joins and set operations [Slides].pdf', 'total_pages': 17, 'page': 0, 'page_label': '1'}, page_content='Please do not copy without permission. © ExploreAI 2023.\nJoins and set operations  \nJoins and unions'),
 Document(metadata={'producer': 'PyPDF', 'creator': 'Google', 'creationdate': '', 'title': 'Joins and set operations [Slides]', 'source': 'C:\\Users\\user\\Downloads\\Joins and set operations [Slides].pdf', 'total_pages': 17, 'page': 1, 'page_label': '2'}, page_content='Introduction\nJoins and unions\nTypes of set operations: \n● UNION \n● INTERSECT  \n● EXCEPT\nTypes of join operations:\n● INNER JOIN\n● LEFT JOIN (or LEFT OUTER JOIN) \n● RIGHT JOIN (or RIGHT OUTER JOIN) \n● FULL JOIN (or FULL OUTER JOIN)\n2\nJoin operations\nJoins are used to combine data from two or more \ntables based on related column(s) between

In [24]:
from langchain_community.vectorstores import FAISS, chroma
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

In [25]:
vectorstore = FAISS.from_documents(documents, embeddings)
vectorstore

In [26]:
#### query 
query = "Explain Inner joins"
documents = vectorstore.similarity_search(query, k=2)
print(documents[0].page_content)

|
An INNER JOIN is a type of join that returns only the rows from both tables where there is a 
match between the specified columns in each table. It filters out the rows that do not have 
corresponding matches in both tables.
Joins and unions
4
INNER JOIN
SELECT 
columns
FROM 
table1
INNER JOIN
table2 
ON
table1.column_name = table2.column_name;
Keyword that specifies the join condition, which is used 
to match rows from table1 with rows from table2 
based on the specified column(s). Matching rows


In [28]:
retriever=vectorstore.as_retriever()
docs=retriever.invoke(query)
print(docs[0].page_content)

|
An INNER JOIN is a type of join that returns only the rows from both tables where there is a 
match between the specified columns in each table. It filters out the rows that do not have 
corresponding matches in both tables.
Joins and unions
4
INNER JOIN
SELECT 
columns
FROM 
table1
INNER JOIN
table2 
ON
table1.column_name = table2.column_name;
Keyword that specifies the join condition, which is used 
to match rows from table1 with rows from table2 
based on the specified column(s). Matching rows


In [29]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnablePassthrough

In [30]:
from langchain_core.runnables import RunnableLambda

retriever_rag = retriever | RunnableLambda(format_docs)
print(retriever_rag.invoke(query))

|
An INNER JOIN is a type of join that returns only the rows from both tables where there is a 
match between the specified columns in each table. It filters out the rows that do not have 
corresponding matches in both tables.
Joins and unions
4
INNER JOIN
SELECT 
columns
FROM 
table1
INNER JOIN
table2 
ON
table1.column_name = table2.column_name;
Keyword that specifies the join condition, which is used 
to match rows from table1 with rows from table2 
based on the specified column(s). Matching rows

Introduction
Joins and unions
Types of set operations: 
● UNION 
● INTERSECT  
● EXCEPT
Types of join operations:
● INNER JOIN
● LEFT JOIN (or LEFT OUTER JOIN) 
● RIGHT JOIN (or RIGHT OUTER JOIN) 
● FULL JOIN (or FULL OUTER JOIN)
2
Join operations
Joins are used to combine data from two or more 
tables based on related column(s) between them. The 
result of a join is a new table that includes columns 
from both tables, arranged side by side.
Set operations
Set operations are used to combine

In [31]:
retriever_rag

VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001682F8B4E10>, search_kwargs={})
| RunnableLambda(format_docs)

In [36]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 1. Prompt
prompt = ChatPromptTemplate.from_template("""
Answer the question based only on the context below and ensure to give table displayed if possible:

<context>
{context}
</context>
                                          
Question: {input}
""")

output_parser = StrOutputParser()

document_chain = prompt | model | output_parser

def chain_():
    chain = (RunnableParallel(
        input=RunnablePassthrough(), 
        docs=retriever
    ) |
    RunnableLambda(lambda x: {
        "input": x["input"],
        "docs": x["docs"],  # RAW Document objects
        "context": format_docs(x["docs"]),  # string version
        "answer": document_chain.invoke({
            "input": x["input"],
            "context": format_docs(x["docs"])
           })
        })
    )
    return chain  

chain_().invoke(
    "Explain Inner joins with examples"
)

{'input': 'Explain Inner joins with examples',
 'docs': [Document(id='d86bff25-8b13-4591-a138-f4ca0e2b2b52', metadata={'producer': 'PyPDF', 'creator': 'Google', 'creationdate': '', 'title': 'Joins and set operations [Slides]', 'source': 'C:\\Users\\user\\Downloads\\Joins and set operations [Slides].pdf', 'total_pages': 17, 'page': 3, 'page_label': '4'}, page_content='|\nAn INNER JOIN is a type of join that returns only the rows from both tables where there is a \nmatch between the specified columns in each table. It filters out the rows that do not have \ncorresponding matches in both tables.\nJoins and unions\n4\nINNER JOIN\nSELECT \ncolumns\nFROM \ntable1\nINNER JOIN\ntable2 \nON\ntable1.column_name = table2.column_name;\nKeyword that specifies the join condition, which is used \nto match rows from table1 with rows from table2 \nbased on the specified column(s). Matching rows'),
  Document(id='2d1d5a28-5d5a-419d-a448-411cc4554a80', metadata={'producer': 'PyPDF', 'creator': 'Google', 

In [37]:
print(chain_().invoke(
    "Explain Inner joins with examples"
)['answer'])

An INNER JOIN is a type of database operation used to combine rows from two or more tables based on a related column between them. It returns only the rows where there is a match between the specified columns in each table, effectively filtering out the rows that do not have corresponding matches in both tables.

For example, consider two tables: `Countries` and `Water_usage`. To create a dataset with columns `Country_name`, `Population`, and `Water_consumed_m3`, containing only the information for countries where data is available in both tables, we would use an INNER JOIN. Here is how the query and its output would look:

```sql
SELECT 
    Countries.Country_name, 
    Countries.Population, 
    Water_usage.Water_consumed_m3 
FROM 
    Countries
INNER JOIN 
    Water_usage 
ON 
    Countries.Country_id = Water_usage.Country_id;
```

**Query Output:**

| Country_name | Population | Water_consumed_m3 |
|--------------|------------|-------------------|
| Country A    | 1,000,000  | 500,